<a href="https://colab.research.google.com/github/nadjib-ken/T7Vr2-B5/blob/main/T7Vr2-B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 0: Problem Definition & Hypotheses (Pair T7Vr2-B5)

1. Problem Definition
We are using a Decision Tree model to predict the severity of road accidents in Addis Ababa (Slight, Serious, or Fatal) based only on the pre-crash circumstances (weather, road, driver, and vehicle info), excluding post-crash casualty variables to prevent data leakage. We will fix random_state = 42 everywhere so our results are reproducible.

2. Hypotheses (Our Predictions Before Testing)
- Hypothesis 1 (Scaling): Scaling the data (StandardScaler) will not change our Decision Tree's score, because Decision Trees use simple if/else rules instead of distances.
- Hypothesis 2 (Overfitting): Making the tree too deep (max_depth = None) will hurt test performance because the tree will memorize noise instead of learning general rules.
- Hypothesis 3 (Rare Fatal Crashes): Because fatal accidents are rare compared to slight injuries, the model will miss most of them unless we use class_weight='balanced' or resampling.

In [1]:
import random
import numpy as np
import pandas as pd

# Step 0: Reproducibility Seed
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Step 1: Load Data & EDA
df = pd.read_csv("Addis_Ababa_city_RTA.csv")

print("1. SHAPE:", df.shape)
print("\n2. DATA TYPES:\n", df.dtypes)

print("\n3. MISSING VALUES (NaN):")
missing = df.isnull().sum()
print(missing[missing > 0])

print("\n4. UNKNOWN / NA TEXT VALUES:")
for col in df.columns:
    unk = df[col].astype(str).str.strip().str.lower().isin(["unknown", "na"]).sum()
    if unk > 0:
        print(f"  {col}: {unk}")

print("\n5. DUPLICATE ROWS:", df.duplicated().sum())

target = "Accident_severity"
print("\n6. TARGET DISTRIBUTION (CLASS BALANCE):")
print(pd.DataFrame({
    "Count": df[target].value_counts(),
    "Percent": (df[target].value_counts(normalize=True) * 100).round(2)
}))

print("\n7. FEATURE INSIGHTS (Light_conditions vs Severity %):")
print((pd.crosstab(df["Light_conditions"], df[target], normalize="index") * 100).round(2))

print("\n8. NUMBER OF VEHICLES INVOLVED:")
print(df["Number_of_vehicles_involved"].value_counts().sort_index())

leakage_cols = [
    "Number_of_casualties", "Casualty_class", "Casualty_gender",
    "Age_band_of_casualty", "Work_of_casuality", "Fitness_of_casuality"
]
print("\n9. LEAKAGE COLUMNS TO DROP IN STEP 2:", leakage_cols)

1. SHAPE: (13063, 31)

2. DATA TYPES:
 Time                           object
Day_of_week                    object
Age_band_of_driver             object
Drivers_gender                 object
Educational_level              object
Vehicle_driver_relation        object
Driving_experience             object
Type_of_vehicle                object
Owner_of_vehicle               object
Service_year_of_vehicle        object
Defect_of_vehicle              object
Area_accident_occured          object
Lanes_or_Medians               object
Road_allignment                object
Types_of_Junction              object
Road_surface_type              object
Road_surface_conditions        object
Light_conditions               object
Weather_conditions             object
Type_of_collision              object
Number_of_vehicles_involved     int64
Number_of_casualties            int64
Vehicle_movement               object
Casualty_class                 object
Casualty_gender                object
Age_band_of